In [ ]:
def template():

    return {
        "attribute_type": "",
        "src":"",    
        "pred":"",     
        "alt":"",    
        "image":"",   
        "rephrase_image":"",
        "gen_img":"",
        "gen1_q":"",
        "gen1_a":"",
        "gen2_q_1":"",
        "gen2_q_2":"",
        "gen2_a":"",
        "loc_in_q":"",
        "loc_out_q":""
    }

In [ ]:
import json

scene = {}
with open("/root/autodl-tmp/mllm_ke/AKE-main/data/clevr_dataset/CLEVR_CoGenT_v1.0/scenes/CLEVR_trainA_scenes.json", "r", encoding="utf-8") as f:
    scene.update(json.load(f))

In [ ]:
from collections import Counter
from tqdm.notebook import tqdm

objects = []
for s in tqdm(scene['scenes']):
    color = [t['color'] for t in s['objects']]
    shape = [t['shape'] for t in s['objects']]
    size = [t['size'] for t in s['objects']]
    material = [t['material'] for t in s['objects']]

    # 场景必须具有多样性
    if len(set(color)) <= 1 or len(set(shape)) <= 1 or len(set(material)) <= 1:
        continue
        
    # 每个object可以被 (color + shape) 唯一识别
    img_obj = [' '.join([obj['color'], obj['shape']]) for obj in s['objects']]
    img_obj = Counter(img_obj)

    # 除去编辑属性 (size+shape / size+color) 能够被唯一识别 
    img_obj_size_shape = [' '.join([obj['size'], obj['shape']]) for obj in s['objects']]
    img_obj_size_shape = Counter(img_obj_size_shape)
    img_obj_size_color = [' '.join([obj['size'], obj['color']]) for obj in s['objects']]
    img_obj_size_color = Counter(img_obj_size_color)
    
    for obj, count in img_obj.items():
        if count == 1:  # 只需判断一次
            target_color, target_shape = obj.split(' ')
            
            # 检查是否唯一，并同时记录两个属性
            color_unique = Counter(color)[target_color] == 1
            shape_unique = Counter(shape)[target_shape] == 1
            
            # 构建基础数据
            base_data = {
                "image": '/'.join([s['image_filename'].split('_')[1], s['image_filename']]),
                "color": color,
                "shape": shape,
                "size": size,
                "material": material,
                "object": obj.split(' ')
            }
            
            # 根据唯一性添加对应的记录
            if color_unique:
                idx = base_data['color'].index(base_data['object'][0])
                size_shape = ' '.join([base_data['size'][idx], base_data['shape'][idx]])
                if img_obj_size_shape[size_shape] == 1:
                    objects.append({**base_data, "attribute_type": "color", "query_attr": size_shape})
            if shape_unique:
                idx = base_data['shape'].index(base_data['object'][1])
                size_color = ' '.join([base_data['size'][idx], base_data['color'][idx]])
                if img_obj_size_color[size_color] == 1:
                    objects.append({**base_data, "attribute_type": "shape", "query_attr": size_color})
len(objects)

In [ ]:
from collections import Counter
tmp = []
for obj in objects:
    tmp.append(obj['attribute_type'])
Counter(tmp)

In [ ]:
color_all = ["gray","blue","brown","yellow","red","green","purple","cyan"]
shape_all = ["cube","cylinder","sphere"]
# forbidden attribute
map_obj = [('green','cyan'),('cyan','green'),('brown','yellow'),('yellow','brown')]

In [ ]:
objects[0]

# Add Gen

In [ ]:
import random
from collections import defaultdict

samples = []

for obj in tqdm(objects):
    rand_num = random.randint(1, 1000000)  

    # 统一处理 color 和 shape 的逻辑
    attr_type = obj['attribute_type']
    
    # 根据类型选择循环的集合
    if attr_type == 'color':
        attr_values = color_all
        src_template = f"What is the color of the {obj['query_attr']}?"
        pred = obj['object'][0]
    else:  # shape
        attr_values = shape_all
        src_template = f"What is the shape of the {obj['query_attr']} object?"
        pred = obj['object'][1]
    
    for attr_value in attr_values:
        
        sample = template()
        
        sample['attribute_type'] = obj['attribute_type']
        sample['image'] = obj['image']
        
        # 目标值不存在
        if attr_value in obj[attr_type]:  # obj['color'] 或 obj['shape']
            continue
        if (attr_value, pred) in map_obj:
            continue
            
        sample['src'] = src_template
        sample['pred'] = pred  
        sample['alt'] = attr_value
        
        # rephrase object - 使用预建立的索引
        flag1 = 0
        for obj_rephrased in objects:
            # 改变属性不搭 或者 目标值不存在
            if obj['attribute_type']!=obj_rephrased['attribute_type'] and attr_value in obj_rephrased[attr_type]:  # obj['color'] 或 obj['shape']
                continue
            if obj_rephrased['object'] == obj['object'] and obj_rephrased['image'] != obj['image']:
                obj_rephrased_idx = obj_rephrased[attr_type].index(pred)
                obj_idx = obj[attr_type].index(pred)
                if obj_rephrased['size'][obj_rephrased_idx] == obj['size'][obj_idx] and obj_rephrased['material'][obj_rephrased_idx] == obj['material'][obj_idx]:
                    sample['rephrase_image'] = obj_rephrased['image']
                    flag1 = 1
                    break
        
        # gen_object - 使用索引优化查找
        cnt = 0
        for obj_gen in objects:
            flag2 = 0
            # 改变属性不搭 或者 目标值不存在
            if obj['attribute_type'] != obj_gen['attribute_type'] and attr_value in obj_gen[attr_type]:  # obj['color'] 或 obj['shape']
                continue

            is_even = rand_num % 2 == 0
            sample['gen_img'] = obj_gen['image'] 
            sample['gen1_a'] = "yes" if is_even else "no"

            # 不是同一个object
            obj_gen_idx = obj_gen[attr_type].index(obj_gen['object'][0 if attr_type=='color' else 1])
            if attr_type=='color' and obj_gen['object'][0] == obj['object'][0] and obj_gen['object'][1] != obj['object'][1] and attr_value not in Counter(obj_gen['color']):
                flag2 = 1
                sample['gen1_q'] = f"Are there any {attr_value if is_even else pred} objects?"
                sample['gen2_q_1'] = f"What is the material of the {attr_value} object? Is it more like rubber or metal? Please output 'None' if there is no {attr_value} object."
                sample['gen2_q_2'] = f"What is the material of the {attr_value} object? Is it more like metal or rubber? Please output 'None' if there is no {attr_value} object."
                sample['gen2_a'] = obj_gen['material'][obj_gen_idx]
            if attr_type=='shape' and obj_gen['object'][0] != obj['object'][0] and obj_gen['object'][1] == obj['object'][1] and attr_value not in Counter(obj_gen['shape']):
                flag2 = 1           
                sample['gen1_q'] = f"Are there any {obj_gen['object'][0]} {attr_value if is_even else pred}s?"
                sample['gen2_q_1'] = f"What is the material of the {attr_value}? Is it more like rubber or metal? Please output 'None' if there is no {attr_value}."
                sample['gen2_q_2'] = f"What is the material of the {attr_value}? Is it more like metal or rubber? Please output 'None' if there is no {attr_value}."
                sample['gen2_a'] = obj_gen['material'][obj_gen_idx]
                
            if flag1 and flag2:
                cnt += 1
                samples.append(sample)
            if cnt >= 10:
                break
len(samples)

# Add Loc

In [ ]:
import json

with open("/root/autodl-tmp/mllm_ke/AKE-main/data/clevr_dataset/qwenvl_new/train_dataset.json","r",encoding="utf-8") as f:
    dataset = json.load(f)

for data in tqdm(dataset):
    flag = 0
    attr_type = data['attribute_type']
    for obj in objects:
        # if obj['image'] in [data['image'], data['rephrase_image'], data['gen_img']]:
        if obj['image'] in [data['image'], data['rephrase_image']]:
            continue

        # combined attribute
        color_size = [' '.join([size, color]) for color, size in zip(obj['color'], obj['size'])]
        shape_size = [' '.join([size, shape]) for shape, size in zip(obj['shape'], obj['size'])]
        
        loc_idx = obj[attr_type].index(obj['object'][0] if attr_type=='color' else obj['object'][1])
        combined_attr = ' '.join([obj['size'][loc_idx], obj['object'][1] if attr_type=='color' else obj['object'][0]])
        if data['pred'] not in obj['object'] and data['alt'] not in obj['object']:
            if attr_type == 'color' and Counter(shape_size)[combined_attr]==1:
                data['loc_img'] = obj['image']
                data['loc_in_q'] = f"What is the color of the {combined_attr}?" 
                data['loc_out_q'] = f"What is the material of the {' '.join(obj['object'])}?"  
                flag = 1
            elif attr_type == 'shape' and Counter(color_size)[combined_attr]==1:
                data['loc_img'] = obj['image']
                data['loc_in_q'] = f"What is the {attr_type} of the {combined_attr} object?"
                data['loc_out_q'] = f"What is the material of the {' '.join(obj['object'])}?"  
                flag = 1
                break
    if not flag:
        print("None!!!")
    del data['m_loc_q'], data['t_loc_q'], data['m_loc_image']

In [ ]:
import json

with open("/root/autodl-tmp/mllm_ke/AKE-main/data/clevr_dataset/qwenvl/train_dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)

# Statistic.

In [ ]:
from collections import Counter
tmp = []
for obj in samples:
    tmp.append(obj['attribute_type'])
Counter(tmp)

In [ ]:
from PIL import Image
Image.open("/root/autodl-tmp/mllm_ke/AKE-main/data/CLEVR_CoGenT_v1.0/images/valB/CLEVR_valB_000133.png")

In [ ]:
dataset[0]